In [1]:
import os
import json
import nltk
import numpy as np
import faiss
import torch
from datetime import datetime
from dotenv import load_dotenv
from groq import Groq
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from pydantic import BaseModel
from typing import List
from pathlib import Path

load_dotenv()

print("Loading models...")
groq_client  = Groq(api_key=os.getenv("GROQ_API_KEY"))
embed_model  = SentenceTransformer('all-MiniLM-L6-v2')
device       = 0 if torch.cuda.is_available() else -1
nli_model    = pipeline(
    "text-classification",
    model="cross-encoder/nli-deberta-v3-base",
    device=device
)
print("All models loaded ✅")

Loading models...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

All models loaded ✅


In [2]:
# ── DATA MODELS ───────────────────────────────────────────────

class ClaimResult(BaseModel):
    claim_id:        int
    claim_text:      str
    matched_span:    str
    doc_index:       int
    label:           str
    confidence:      float
    retrieval_score: float
    was_reranked:    bool

class VeriFaithResult(BaseModel):
    faithfulness_score:   float
    total_claims:         int
    supported_count:      int
    neutral_count:        int
    contradiction_count:  int
    per_claim_report:     List[ClaimResult]
    contradiction_report: List[ClaimResult]
    low_confidence_flags: List[str]


# ── MODULE 1: CLAIM EXTRACTOR ─────────────────────────────────

def extract_claims(answer: str) -> list[dict]:
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are a claim decomposition engine for AI evaluation.
Break down any given text into individual atomic factual claims.
Rules:
- Each claim must contain ONE fact only
- Each claim must be self-contained and understandable alone
- Each claim must be a declarative statement
- Do NOT include opinions or vague statements
- Do NOT merge two facts into one claim
Return ONLY a valid JSON array. No explanation. No markdown.
Format: [{"claim_id": 1, "text": "..."}, {"claim_id": 2, "text": "..."}]"""
            },
            {
                "role": "user",
                "content": f"Decompose this answer into atomic claims:\n\n{answer}"
            }
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )
    raw    = response.choices[0].message.content.strip()
    raw    = raw.replace("```json", "").replace("```", "").strip()
    parsed = json.loads(raw)
    if isinstance(parsed, dict):
        claims = list(parsed.values())[0]
    else:
        claims = parsed
    return claims


# ── MODULE 2: SPAN RETRIEVER ──────────────────────────────────

def split_into_sentences(docs: list[str]) -> list[dict]:
    all_sentences = []
    for doc_idx, doc in enumerate(docs):
        sentences = nltk.sent_tokenize(doc.strip())
        for sent in sentences:
            sent = sent.strip()
            if len(sent) > 10:
                all_sentences.append({
                    "sentence": sent,
                    "doc_index": doc_idx
                })
    return all_sentences

def build_faiss_index(sentences: list[dict]):
    texts      = [s["sentence"] for s in sentences]
    embeddings = embed_model.encode(texts)
    embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    index      = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings.astype('float32'))
    return index, sentences

def retrieve_span(claim: str, index, sentence_store: list[dict], top_k=5) -> dict:
    claim_emb = embed_model.encode([claim])
    claim_emb = claim_emb / np.linalg.norm(claim_emb, axis=1, keepdims=True)
    scores, indices = index.search(claim_emb.astype('float32'), top_k)
    best = sentence_store[indices[0][0]]
    return {
        "sentence":        best["sentence"],
        "doc_index":       best["doc_index"],
        "similarity_score": round(float(scores[0][0]), 4),
        "low_confidence":  float(scores[0][0]) < 0.70,
        "top_3_spans": [
            {
                "sentence":  sentence_store[indices[0][i]]["sentence"],
                "doc_index": sentence_store[indices[0][i]]["doc_index"],
                "score":     round(float(scores[0][i]), 4)
            }
            for i in range(top_k)
        ]
    }


# ── MODULE 3: ENTAILMENT CHECKER ─────────────────────────────

def check_entailment(claim: str, span: str) -> dict:
    nli_input  = f"{span} [SEP] {claim}"
    result     = nli_model(nli_input)
    label      = result[0]["label"].upper()
    confidence = round(result[0]["score"], 4)
    label_map  = {
        "ENTAILMENT":    "ENTAILMENT",
        "NEUTRAL":       "NEUTRAL",
        "CONTRADICTION": "CONTRADICTION",
        "LABEL_0":       "CONTRADICTION",
        "LABEL_1":       "NEUTRAL",
        "LABEL_2":       "ENTAILMENT"
    }
    return {"label": label_map.get(label, label), "confidence": confidence}

def rerank_with_nli(claim: str, top_spans: list[dict]) -> dict:
    all_results = []
    for rank, span_dict in enumerate(top_spans):
        nli_result = check_entailment(claim, span_dict["sentence"])
        all_results.append({
            "rank":             rank + 1,
            "span":             span_dict["sentence"],
            "doc_index":        span_dict["doc_index"],
            "retrieval_score":  span_dict["score"],
            "nli_label":        nli_result["label"],
            "nli_confidence":   nli_result["confidence"]
        })
    for label in ["CONTRADICTION", "ENTAILMENT", "NEUTRAL"]:
        matches = [r for r in all_results if r["nli_label"] == label]
        if matches:
            best = max(matches, key=lambda x: x["nli_confidence"])
            break
    return {
        "best_span":      best["span"],
        "doc_index":      best["doc_index"],
        "label":          best["nli_label"],
        "confidence":     best["nli_confidence"],
        "retrieval_rank": best["rank"],
        "reranked":       best["rank"] != 1,
        "all_results":    all_results
    }


# ── SCORER ────────────────────────────────────────────────────

def evaluate(answer: str, source_docs: list[str]) -> VeriFaithResult:
    raw_claims                    = extract_claims(answer)
    sentences                     = split_into_sentences(source_docs)
    faiss_index, sentence_store   = build_faiss_index(sentences)
    claim_results                 = []
    low_confidence_flags          = []

    for raw_claim in raw_claims:
        retrieval  = retrieve_span(raw_claim["text"], faiss_index, sentence_store)
        nli_result = rerank_with_nli(raw_claim["text"], retrieval["top_3_spans"])
        if retrieval["low_confidence"]:
            low_confidence_flags.append(raw_claim["text"])
        claim_results.append(ClaimResult(
            claim_id        = raw_claim["claim_id"],
            claim_text      = raw_claim["text"],
            matched_span    = nli_result["best_span"],
            doc_index       = nli_result["doc_index"],
            label           = nli_result["label"],
            confidence      = nli_result["confidence"],
            retrieval_score = retrieval["similarity_score"],
            was_reranked    = nli_result["reranked"]
        ))

    total        = len(claim_results)
    supported    = sum(1 for r in claim_results if r.label == "ENTAILMENT")
    neutral      = sum(1 for r in claim_results if r.label == "NEUTRAL")
    contradicted = sum(1 for r in claim_results if r.label == "CONTRADICTION")

    return VeriFaithResult(
        faithfulness_score   = round(supported / total, 4) if total > 0 else 0.0,
        total_claims         = total,
        supported_count      = supported,
        neutral_count        = neutral,
        contradiction_count  = contradicted,
        per_claim_report     = claim_results,
        contradiction_report = [r for r in claim_results if r.label == "CONTRADICTION"],
        low_confidence_flags = low_confidence_flags
    )

print("All pipeline functions ready ✅")

All pipeline functions ready ✅


In [3]:
def generate_json_report(
    result: VeriFaithResult,
    answer: str,
    source_docs: list[str]
) -> dict:
    """
    Produces a clean, structured JSON report.
    Suitable for API responses and logging.
    """
    
    label_map = {
        "ENTAILMENT":    "supported",
        "NEUTRAL":       "unverified",
        "CONTRADICTION": "contradicted"
    }
    
    report = {
        "verifaith_version": "1.0.0",
        "evaluated_at":      datetime.now().isoformat(),
        
        "summary": {
            "faithfulness_score":  result.faithfulness_score,
            "verdict":             (
                "FAITHFUL"     if result.faithfulness_score >= 0.75 else
                "PARTIAL"      if result.faithfulness_score >= 0.40 else
                "UNFAITHFUL"
            ),
            "total_claims":        result.total_claims,
            "supported":           result.supported_count,
            "unverified":          result.neutral_count,
            "contradicted":        result.contradiction_count,
            "has_contradictions":  result.contradiction_count > 0
        },
        
        "input": {
            "answer":       answer,
            "source_count": len(source_docs)
        },
        
        "claim_breakdown": [
            {
                "claim_id":        r.claim_id,
                "claim":           r.claim_text,
                "status":          label_map[r.label],
                "confidence":      r.confidence,
                "evidence_span":   r.matched_span,
                "source_doc":      r.doc_index,
                "retrieval_score": r.retrieval_score,
                "was_reranked":    r.was_reranked
            }
            for r in result.per_claim_report
        ],
        
        "contradiction_report": [
            {
                "claim":       r.claim_text,
                "conflicts_with": r.matched_span,
                "confidence":  r.confidence,
                "source_doc":  r.doc_index
            }
            for r in result.contradiction_report
        ],
        
        "warnings": {
            "low_confidence_retrievals": result.low_confidence_flags
        }
    }
    
    return report

print("JSON report generator ready ✅")

JSON report generator ready ✅


In [4]:
def generate_text_report(result: VeriFaithResult, answer: str) -> str:
    """
    Produces a clean human-readable report.
    Suitable for printing in terminals and logs.
    """
    
    verdict = (
        "FAITHFUL"   if result.faithfulness_score >= 0.75 else
        "PARTIAL"    if result.faithfulness_score >= 0.40 else
        "UNFAITHFUL"
    )
    
    verdict_emoji = {
        "FAITHFUL":   "✅",
        "PARTIAL":    "⚠️",
        "UNFAITHFUL": "❌"
    }
    
    label_emoji = {
        "ENTAILMENT":    "✅",
        "NEUTRAL":       "⚪",
        "CONTRADICTION": "❌"
    }
    
    lines = []
    lines.append("=" * 60)
    lines.append("         VERIFAITH FAITHFULNESS REPORT")
    lines.append("=" * 60)
    lines.append(f"  Evaluated At : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    lines.append(f"  Answer       : {answer[:80]}...")
    lines.append("")
    
    # Score block
    lines.append("─" * 60)
    lines.append("  SCORE SUMMARY")
    lines.append("─" * 60)
    lines.append(f"  Faithfulness Score : {result.faithfulness_score}")
    lines.append(f"  Verdict            : {verdict_emoji[verdict]} {verdict}")
    lines.append(f"  Total Claims       : {result.total_claims}")
    lines.append(f"  ✅ Supported       : {result.supported_count}")
    lines.append(f"  ⚪ Unverified      : {result.neutral_count}")
    lines.append(f"  ❌ Contradicted    : {result.contradiction_count}")
    lines.append("")
    
    # Per claim breakdown
    lines.append("─" * 60)
    lines.append("  PER CLAIM BREAKDOWN")
    lines.append("─" * 60)
    
    for r in result.per_claim_report:
        emoji = label_emoji[r.label]
        lines.append(f"\n  {emoji} Claim {r.claim_id}: {r.claim_text}")
        lines.append(f"     Status    : {r.label} ({r.confidence})")
        lines.append(f"     Evidence  : {r.matched_span[:80]}...")
        lines.append(f"     Doc Index : {r.doc_index}")
        if r.was_reranked:
            lines.append(f"     Note      : 🔄 Span was reranked by NLI")
    
    # Contradiction report
    if result.contradiction_report:
        lines.append("")
        lines.append("─" * 60)
        lines.append("  🚨 CONTRADICTION REPORT")
        lines.append("─" * 60)
        lines.append("  These claims DIRECTLY CONFLICT with source documents:")
        for r in result.contradiction_report:
            lines.append(f"\n  ❌ Claim    : {r.claim_text}")
            lines.append(f"     Conflicts : {r.matched_span[:80]}...")
            lines.append(f"     Confidence: {r.confidence}")
    
    # Warnings
    if result.low_confidence_flags:
        lines.append("")
        lines.append("─" * 60)
        lines.append("  ⚠️  LOW CONFIDENCE RETRIEVALS")
        lines.append("─" * 60)
        lines.append("  These claims had weak evidence matches:")
        for flag in result.low_confidence_flags:
            lines.append(f"  - {flag}")
    
    lines.append("")
    lines.append("=" * 60)
    
    return "\n".join(lines)

print("Text report generator ready ✅")

Text report generator ready ✅


In [5]:
def generate_markdown_report(
    result: VeriFaithResult,
    answer: str
) -> str:
    """
    Produces a Markdown report.
    Suitable for GitHub READMEs, research papers, documentation.
    """
    
    verdict = (
        "FAITHFUL"   if result.faithfulness_score >= 0.75 else
        "PARTIAL"    if result.faithfulness_score >= 0.40 else
        "UNFAITHFUL"
    )
    
    score_bar_filled = int(result.faithfulness_score * 10)
    score_bar        = "█" * score_bar_filled + "░" * (10 - score_bar_filled)
    
    lines = []
    lines.append("# VeriFaith Evaluation Report")
    lines.append(f"> Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    lines.append("")
    
    # Score summary
    lines.append("## Summary")
    lines.append("")
    lines.append(f"| Metric | Value |")
    lines.append(f"|--------|-------|")
    lines.append(f"| **Faithfulness Score** | `{result.faithfulness_score}` |")
    lines.append(f"| **Verdict** | **{verdict}** |")
    lines.append(f"| Score Bar | `{score_bar}` |")
    lines.append(f"| Total Claims | {result.total_claims} |")
    lines.append(f"| ✅ Supported | {result.supported_count} |")
    lines.append(f"| ⚪ Unverified | {result.neutral_count} |")
    lines.append(f"| ❌ Contradicted | {result.contradiction_count} |")
    lines.append("")
    
    # Answer
    lines.append("## Evaluated Answer")
    lines.append(f"> {answer}")
    lines.append("")
    
    # Per claim table
    lines.append("## Per Claim Breakdown")
    lines.append("")
    lines.append("| ID | Claim | Status | Confidence | Evidence |")
    lines.append("|----|-------|--------|------------|----------|")
    
    label_emoji = {
        "ENTAILMENT":    "✅ Supported",
        "NEUTRAL":       "⚪ Unverified",
        "CONTRADICTION": "❌ Contradicted"
    }
    
    for r in result.per_claim_report:
        evidence_short = r.matched_span[:50] + "..."
        lines.append(
            f"| {r.claim_id} "
            f"| {r.claim_text} "
            f"| {label_emoji[r.label]} "
            f"| {r.confidence} "
            f"| {evidence_short} |"
        )
    lines.append("")
    
    # Contradictions
    if result.contradiction_report:
        lines.append("## 🚨 Contradictions Found")
        lines.append("")
        lines.append("These claims **directly conflict** with the source documents:\n")
        for r in result.contradiction_report:
            lines.append(f"**Claim:** {r.claim_text}  ")
            lines.append(f"**Conflicts with:** {r.matched_span}  ")
            lines.append(f"**Confidence:** {r.confidence}  ")
            lines.append("")
    
    # Warnings
    if result.low_confidence_flags:
        lines.append("## ⚠️ Warnings")
        lines.append("")
        lines.append("Low confidence retrievals — verify manually:\n")
        for flag in result.low_confidence_flags:
            lines.append(f"- {flag}")
    
    return "\n".join(lines)

print("Markdown report generator ready ✅")

Markdown report generator ready ✅


In [6]:
# Test answers from Day 4
faithful_answer = """The Eiffel Tower was built in 1889 and stands 330 metres tall. 
It was designed by Gustave Eiffel and is located in Paris, France."""

hallucinated_answer = """The Eiffel Tower was built in 1950 and stands 500 metres tall.
It was designed by Leonardo da Vinci and is located in London, England."""

source_docs = [
    """The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars 
    in Paris, France. It was constructed between 1887 and 1889 as the centerpiece 
    of the 1889 World's Fair. The tower was designed and built by Alexandre Gustave Eiffel, 
    a French civil engineer. It stands 330 metres tall and is one of the most recognizable 
    structures in the world."""
]

# Run evaluations
print("Running evaluations...")
result_faithful     = evaluate(faithful_answer, source_docs)
result_hallucinated = evaluate(hallucinated_answer, source_docs)
print("Done ✅")

Running evaluations...
Done ✅


In [7]:
# ── TEXT REPORTS ──────────────────────────────────────────────
print(generate_text_report(result_faithful, faithful_answer))
print("\n\n")
print(generate_text_report(result_hallucinated, hallucinated_answer))

         VERIFAITH FAITHFULNESS REPORT
  Evaluated At : 2026-04-24 12:35:44
  Answer       : The Eiffel Tower was built in 1889 and stands 330 metres tall. 
It was designed ...

────────────────────────────────────────────────────────────
  SCORE SUMMARY
────────────────────────────────────────────────────────────
  Faithfulness Score : 0.5
  Verdict            : ⚠️ PARTIAL
  Total Claims       : 4
  ✅ Supported       : 2
  ⚪ Unverified      : 2
  ❌ Contradicted    : 0

────────────────────────────────────────────────────────────
  PER CLAIM BREAKDOWN
────────────────────────────────────────────────────────────

  ⚪ Claim 1: The Eiffel Tower was built in 1889
     Status    : NEUTRAL (0.9997)
     Evidence  : The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars 
 ...
     Doc Index : 0
     Note      : 🔄 Span was reranked by NLI

  ⚪ Claim 2: The Eiffel Tower stands 330 metres tall
     Status    : NEUTRAL (0.9998)
     Evidence  : The Eiffel Tower is a wrought

In [8]:
json_report = generate_json_report(result_hallucinated, hallucinated_answer, source_docs)
print(json.dumps(json_report, indent=2))

{
  "verifaith_version": "1.0.0",
  "evaluated_at": "2026-04-24T12:35:55.014536",
  "summary": {
    "faithfulness_score": 0.5,
    "verdict": "PARTIAL",
    "total_claims": 4,
    "supported": 2,
    "unverified": 2,
    "contradicted": 0,
    "has_contradictions": false
  },
  "input": {
    "answer": "The Eiffel Tower was built in 1950 and stands 500 metres tall.\nIt was designed by Leonardo da Vinci and is located in London, England.",
    "source_count": 1
  },
  "claim_breakdown": [
    {
      "claim_id": 1,
      "claim": "The Eiffel Tower was built in 1889",
      "status": "unverified",
      "confidence": 0.9997,
      "evidence_span": "The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars \n    in Paris, France.",
      "source_doc": 0,
      "retrieval_score": 0.7728,
      "was_reranked": true
    },
    {
      "claim_id": 2,
      "claim": "The Eiffel Tower stands 324 metres tall",
      "status": "unverified",
      "confidence": 0.9998,
      "

In [11]:
# Create reports directory
reports_dir = Path("reports")
reports_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save JSON - faithful
json_path = reports_dir / f"report_faithful_{timestamp}.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        generate_json_report(result_faithful, faithful_answer, source_docs),
        f, indent=2, ensure_ascii=False
    )

# Save Markdown - faithful
md_path = reports_dir / f"report_faithful_{timestamp}.md"
with open(md_path, "w", encoding="utf-8") as f:
    f.write(generate_markdown_report(result_faithful, faithful_answer))

# Save JSON - hallucinated
json_path2 = reports_dir / f"report_hallucinated_{timestamp}.json"
with open(json_path2, "w", encoding="utf-8") as f:
    json.dump(
        generate_json_report(result_hallucinated, hallucinated_answer, source_docs),
        f, indent=2, ensure_ascii=False
    )

# Save Markdown - hallucinated
md_path2 = reports_dir / f"report_hallucinated_{timestamp}.md"
with open(md_path2, "w", encoding="utf-8") as f:
    f.write(generate_markdown_report(result_hallucinated, hallucinated_answer))

print(f"Reports saved to /reports folder:")
print(f"  ✅ {json_path.name}")
print(f"  ✅ {md_path.name}")
print(f"  ✅ {json_path2.name}")
print(f"  ✅ {md_path2.name}")

Reports saved to /reports folder:
  ✅ report_faithful_20260424_124431.json
  ✅ report_faithful_20260424_124431.md
  ✅ report_hallucinated_20260424_124431.json
  ✅ report_hallucinated_20260424_124431.md


In [ ]:
print("""
✅ Day 5 Complete!

Copy report functions into verifaith/report.py:
  → generate_json_report()
  → generate_text_report()
  → generate_markdown_report()

Project Structure Now:
  ✅ verifaith/claim_extractor.py
  ✅ verifaith/span_retriever.py
  ✅ verifaith/entailment_checker.py
  ✅ verifaith/scorer.py
  ✅ verifaith/report.py            ← today
  ✅ verifaith/models.py
  📁 reports/                       ← saved report files

Day 6 → FastAPI wrapper
        One endpoint: POST /evaluate
        Input:  answer + source_docs
        Output: full VeriFaith JSON report
""")